# Run Presidio
Default  Spacy (en_core_web_lg) NER 

## Set Up

### Import packages

In [ ]:
# Install packages if not available on your cluster
import pandas as pd
from pyspark.sql.functions import pandas_udf, PandasUDFType, col
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from pyspark.sql.types import StructType, StructField, StringType
import spacy

In [ ]:
# Check that model can be loaded and has expected labels
nlp = spacy.load("en_core_web_lg")
nlp.get_pipe("ner").labels

### Spark Setup

In [ ]:
# Configure Spark session
spark.conf.set("spark.sql.shuffle.partitions", "400")
spark.conf.set("spark.sql.execution.arrow.enabled", "true")
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "true")

sc = spark.sparkContext

In [ ]:
# Review configuration details
for item in spark.sparkContext.getConf().getAll():
    print(item)

### Set parameters

See available Presidio entity types: https://microsoft.github.io/presidio/supported_entities/

In [ ]:
data_path = "catalog.schema.table"
data = spark.sql("""select * from """ + data_path)
target_col = "MY_TARGET"
output_table_name = "catalog.schema.output_table"
access_group = "group-name"

In [ ]:
my_entities_list = [
    "PERSON",
    "NRP",
    "DATE_TIME",
    "LOCATION",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "US_SSN",
    "ORGANIZATION",
]  # Adjust as needed
my_allow_list = []  # Terms for the model NOT to mark as PII
my_deny_list = []  # Terms for the model to ALWAYS mark as PII

## Run Presidio Analyzer

In [ ]:
# Initialize Presidio's AnalyzerEngine
analyzer = AnalyzerEngine()

# broadcast the engines to the cluster nodes
broadcasted_analyzer = sc.broadcast(analyzer)


# Define a function that processes a Pandas Series, returns a Pandas DataFrame
def find_pii_entities(text: pd.Series) -> pd.DataFrame:

    analyzer = broadcasted_analyzer.value
    results = []

    # If using a deny list, add that as a category used by the analyzer
    if len(my_deny_list) > 0:
        pii_signal_recognizer = PatternRecognizer(
            supported_entity="PII_SIGNAL", deny_list=my_deny_list
        )
        analyzer.registry.add_recognizer(pii_signal_recognizer)
        my_entities_list.append("PII_SIGNAL")

    for t in text:
        # Enter parameters for analyzer
        analyzer_results = analyzer.analyze(
            text=t,
            language="en",
            entities=my_entities_list,
            allow_list=my_allow_list,
        )

        # Extract detected entities
        detected_entities = ", ".join(
            [
                f"{res.entity_type}: {res.start}-{res.end}"
                for res in analyzer_results
            ]
        )

        # Extract flagged text
        flagged_text = ", ".join(
            [t[res.start : res.end] for res in analyzer_results]
        )

        results.append((detected_entities, flagged_text))

    return pd.DataFrame(results, columns=["detected_entities", "flagged_text"])


# Define the function as a Pandas UDF with a StructType return type
schema = StructType(
    [
        StructField("detected_entities", StringType(), True),
        StructField("flagged_text", StringType(), True),
    ]
)

detect_pii = pandas_udf(find_pii_entities, returnType=schema)

In [ ]:
results_df = data.repartition(64).withColumn(
    "PII_RESULTS", (detect_pii(col(target_col)))
)

In [ ]:
# Split the resulting column into separate columns
results_df = results_df.select(
    "*",
    results_df.PII_RESULTS["detected_entities"].alias("detected_entities"),
    results_df.PII_RESULTS["flagged_text"].alias("flagged_text"),
)

## Preview Results

In [ ]:
results_df.show()

In [ ]:
# Check row count in output
spark.sql("select count(*) from {df}", df=results_df).show()

## Save Results

In [ ]:
results_df.write.mode("overwrite").saveAsTable(output_table_name)

In [ ]:
spark.sql("GRANT SELECT ON " + output_table_name + " TO " + access_group)